In [14]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import joblib

# Load dataset
X, y = load_iris(return_X_y=True)

# Train model
model = LogisticRegression(max_iter=200)
model.fit(X, y)

# Save model
joblib.dump(model, "model.pkl")

print("Model saved")

Model saved


In [15]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Model

# connect to workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# register model
model = ml_client.models.create_or_update(
    Model(
        name="iris-model",
        path="model.pkl",
        type="custom_model"
    )
)

print("Model registered")

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading model.pkl (< 1 MB): 100%|██████████| 991/991 [00:00<00:00, 65.4kB/s]




Model registered


In [17]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="sklearn-env",
    conda_file="environment.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
)

ml_client.environments.create_or_update(env)

print("Environment created")

Environment created


In [18]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name="iris-endpoint",
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

print("Endpoint created")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


Endpoint created


In [20]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="iris-endpoint",
    model="iris-model:1",
    environment="sklearn-env@latest",
    code_configuration=CodeConfiguration(
        code="./",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1
)

ml_client.begin_create_or_update(deployment).result()

print("Deployment created")

Check: endpoint iris-endpoint exists
Uploading vishesh059 (0.03 MBs): 100%|██████████| 25351/25351 [00:00<00:00, 557097.20it/s]




HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"VmSize\":[\"Not enough quota available for Standard_DS3_v2 in SubscriptionId 440cbe94-6375-4d51-9b90-dced7dbdca5e. Current usage/limit: 4/6. Additional needed: 8 Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-1956a36d2db3891f1e1bb028f9f6531b-a79b3bc3109af03b-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"VmSize\":[\"Not enough quota available for Standard_DS3_v2 in SubscriptionId 440cbe94-6375-4d51-9b90-dced7dbdca5e. Current usage/limit: 4/6. Additional needed: 8 Please see troubleshooting guide, available here: https://aka.

In [21]:
endpoint = ml_client.online_endpoints.get("iris-endpoint")

endpoint.traffic = {"blue": 100}

ml_client.begin_create_or_update(endpoint).result()

print("Traffic routed to deployment")

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"DeploymentWeights\":[\"Deployments given positive traffic values should be either in a successful or failed state. Unmatched deployments: [blue]\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-0448ac5c4faf60412098ca25417de6b9-57d1db0c1a346332-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"DeploymentWeights\":[\"Deployments given positive traffic values should be either in a successful or failed state. Unmatched deployments: [blue]\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-0448ac5c4faf60412098ca25417de6b

In [22]:
endpoint = ml_client.online_endpoints.get("iris-endpoint")

print("Endpoint URL:", endpoint.scoring_uri)

Endpoint URL: https://iris-endpoint.eastasia.inference.ml.azure.com/score
